# ML Study Tracker: Unsupervised Learning — Practical Use Cases (v3, Beginner-Annotated)

**Unsupervised learning** means there is **no target column `y`** — no labels, no "right answers." The algorithm's job is to discover structure hidden in the features alone. The three big families, all covered here:

| Family | Question it answers | Sections |
|---|---|---|
| **Clustering** | "Which rows naturally belong together?" | 1–4 (K-Means, Hierarchical, DBSCAN, GMM) |
| **Dimensionality reduction** | "Can we compress many features into few, keeping the essence?" | 5–6 (PCA, t-SNE) |
| **Anomaly detection** | "Which rows don't belong at all?" | 7 (Isolation Forest) |

### The central difficulty: how do you know it worked?
In supervised learning, the test set grades you. Here there's no answer key, so we lean on:
- **Internal validation** — do clusters look tight and well-separated *by the data's own geometry*? (inertia, **silhouette score**, Davies-Bouldin)
- **Stability** — does the structure survive perturbing the data? (v3's Section 1 Example B makes this concrete — it's the closest thing unsupervised learning has to a validation set)
- **Model-selection criteria** — statistical scores like **BIC** that reward fit but penalize complexity.
- **External validation** — *only in study settings*: when true labels secretly exist (wine cultivars, digit identities), we can compare clusters to them with **Adjusted Rand Index (ARI)**. ARI = 1 means clusters match labels perfectly, ARI ≈ 0 means no better than random assignment. In real unsupervised work you don't have this luxury — we use it here purely to check our understanding.
- **Downstream usefulness** — did the segments/components actually help the business or the next model? The ultimate judge.

### How to use this notebook
1. Run the **Common Imports** cell first.
2. Every numbered section is **self-contained** — run any one independently after imports. (Within a section, Example B reuses Example A's data — run A before B.)
3. Read the concept markdown before the code, run **Example A**, read the **"Reading the output"** notes, then run **Example B** (a second angle — usually a heuristic v2 only *mentioned*, or a failure mode v2 only *warned about*, now demonstrated).
4. Finish each section with the **🧠 Can you answer?** questions. They are deliberately hard. If you can't defend an answer *out loud, without looking*, you haven't finished the section. Write your answers in the Notes cell.
5. All datasets here are bundled with sklearn or synthesized locally — **no internet needed**, unlike the Supervised notebook.

### What changed from v2
- **Audit result first:** unlike the Supervised notebook (whose v2 tuned on the test set in two places), this notebook's v2 was methodologically sound — every number reproduces. So v3's changes are additions, not corrections.
- **Every "try it yourself" from v2 is now implemented** as an Example B: the k-distance elbow heuristic (S3), the perplexity sensitivity check (S6), the contamination sweep (S7).
- **New honest-lesson demos:** cluster stability analysis (S1), single-linkage chaining failure (S2), DBSCAN's varying-density blind spot (S3), covariance-type selection on stretched clusters (S4), PCA as a compressor you can *see* (S5), and a subtle-fraud scenario where Isolation Forest's perfect scores collapse (S7).
- **🧠 Can you answer?** interrogation blocks per section.

### Key vocabulary used throughout
- **Silhouette score** (−1 to +1): for each point, compares its distance to its own cluster vs. the nearest other cluster. Near +1 = snugly inside its cluster; near 0 = sitting on a boundary; negative = probably in the wrong cluster. We report the average.
- **Inertia**: sum of squared distances from points to their cluster center. Lower = tighter, but it ALWAYS decreases as you add clusters — hence the "elbow" heuristic rather than simple minimization.
- **Stability (ARI between refits)**: refit on perturbed/resampled data and measure agreement with the original solution. Structure that vanishes under resampling was noise, not signal.

## Table of Contents
- [Common Imports & Configuration](#Common-Imports-&-Configuration)
- [1. K-Means — Customer Segmentation](#1.-K-Means-—-Customer-Segmentation)
- [2. Hierarchical Clustering — Dendrogram Analysis](#2.-Hierarchical-Clustering-—-Dendrogram-Analysis)
- [3. DBSCAN — Density Clusters & Noise](#3.-DBSCAN-—-Density-Clusters-&-Noise)
- [4. Gaussian Mixture Models — Soft Clustering & BIC](#4.-Gaussian-Mixture-Models-—-Soft-Clustering-&-BIC)
- [5. PCA — Dimensionality Reduction & Variance](#5.-PCA-—-Dimensionality-Reduction-&-Variance)
- [6. t-SNE — Non-Linear Visualization](#6.-t-SNE-—-Non-Linear-Visualization)
- [7. Isolation Forest — Anomaly Detection](#7.-Isolation-Forest-—-Anomaly-Detection)
- [Mini Project Tracker](#Mini-Project-Tracker)
- [Experiment Log](#Experiment-Log)

In [ ]:
# ============================================================
# Common Imports & Configuration
# Run this cell FIRST. Every section below assumes these exist.
# ============================================================
import numpy as np                     # numerical arrays and math
import pandas as pd                    # tabular data (DataFrames)
import matplotlib.pyplot as plt        # plotting

from sklearn.preprocessing import StandardScaler   # mean 0, std 1 per feature
from sklearn.pipeline import Pipeline

# Validation metrics for the label-free world:
#   silhouette_score      -> internal: cohesion vs separation, -1..+1
#   davies_bouldin_score  -> internal: avg cluster similarity, LOWER is better
#   adjusted_rand_score   -> external: agreement with true labels (study use
#                            only) — and, in v3, agreement between two
#                            clusterings of the same data (stability!)
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

RANDOM_SEED = 42
# As in the Supervised notebook: no global np.random.seed(). Each section makes
# its own rng = np.random.default_rng(RANDOM_SEED) so every cell reproduces
# identically regardless of execution order.

## 1. K-Means — Customer Segmentation
**Dataset Target:** Synthetic RFM-Style Customer Behavior (Recency, Frequency, Monetary)

### The concept, in plain words
K-Means partitions data into **k** groups by repeating two steps until nothing moves (Lloyd's algorithm):
1. **Assign** each point to its nearest cluster center (centroid).
2. **Update** each centroid to the mean of its assigned points.

It's fast, simple, and the default first clustering attempt. Its assumptions — and therefore its failure modes — are: clusters are **round-ish, similar-sized blobs**, and **you must pick k in advance**.

### The business framing: RFM
Retail segmentation classically uses **R**ecency (days since last purchase — lower is better), **F**requency (purchases per period), **M**onetary (spend). We synthesize three personas — *Champions*, *Regulars*, *Dormant* — and check whether K-Means rediscovers them from the raw numbers alone.

### Picking k: two lenses, used together
- **Elbow (inertia) curve:** inertia always falls as k grows; look for the "elbow" where the drop suddenly flattens — extra clusters past that point buy little tightness.
- **Silhouette score:** unlike inertia it can *peak*, giving an actual argmax. When elbow and silhouette agree, be confident; when they disagree, profile both candidate k's and let interpretability decide.

**v3 adds a third lens — stability (Example B):** if the same k, refit on bootstrap resamples of the data, keeps producing the *same* clusters, the structure is real; if solutions reshuffle from resample to resample, you're clustering noise. This is the closest unsupervised learning gets to a validation set.

⚠️ Two practical gotchas baked into the code: **standardize first** (Monetary is in hundreds, Recency in days — unscaled distance would be all about money), and use `n_init=10` (K-Means can converge to a poor local optimum from a bad random start; running 10 starts and keeping the best is cheap insurance).

### Study Checklist
- [ ] Centroid Allocation Optimization Loops (Lloyd's algorithm intuition)
- [ ] Euclidean Distance Scaling Dependency (standardize first, always)
- [ ] Elbow Curve via Inertia — and why it's ambiguous alone
- [ ] Silhouette Score Cohesion Evaluation
- [ ] Bootstrap Stability Analysis (Example B — the third lens on k)
- [ ] Post-Clustering Business Profiling (translate clusters into personas)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: K-Means Segmentation, Elbow + Silhouette
# ============================================================
from sklearn.cluster import KMeans

rng = np.random.default_rng(RANDOM_SEED)

# --- Step 1: synthesize customers from three KNOWN personas ---
# Because we planted the personas, we can verify K-Means recovers them.
# Columns: Recency (days since last purchase), Frequency (orders/yr), Monetary ($/yr)
n_per = 300
champions = np.column_stack([rng.normal(5, 2, n_per),      # bought very recently
                             rng.normal(40, 8, n_per),     # buy often
                             rng.normal(900, 150, n_per)]) # spend a lot
regulars  = np.column_stack([rng.normal(20, 5, n_per),
                             rng.normal(12, 4, n_per),
                             rng.normal(300, 80, n_per)])
dormant   = np.column_stack([rng.normal(120, 30, n_per),   # haven't bought in months
                             rng.normal(2, 1, n_per),
                             rng.normal(60, 30, n_per)])
df = pd.DataFrame(np.vstack([champions, regulars, dormant]),
                  columns=['Recency_Days', 'Frequency', 'Monetary'])
# NOTE: no labels are kept — from here on, the algorithm is on its own.

# --- Step 2: standardize (Monetary ~ hundreds would dominate Euclidean distance) ---
X_scaled = StandardScaler().fit_transform(df)

# --- Step 3: choose k — run K-Means for k=2..7, record both criteria ---
ks, inertias, sils = range(2, 8), [], []
for k in ks:
    # n_init=10: run from 10 random starts, keep the best (avoids bad local optima)
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, km.labels_))
    print(f"k={k} | inertia={km.inertia_:>8.1f} | silhouette={sils[-1]:.4f}")

# Plot both criteria side by side — the standard 'choosing k' picture
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(ks), inertias, marker='o'); ax[0].set(title='Elbow (inertia)', xlabel='k')
ax[1].plot(list(ks), sils, marker='o');     ax[1].set(title='Silhouette (peak = best)', xlabel='k')
plt.tight_layout(); plt.show()

# --- Step 4: fit final k=3 and PROFILE clusters in ORIGINAL units ---
# groupby-mean in raw units is what turns math clusters into business personas.
km = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_SEED).fit(X_scaled)
df['Cluster'] = km.labels_
print("\nCluster personas (means in original units):")
print(df.groupby('Cluster').mean().round(1))
print("\nCluster sizes:", df['Cluster'].value_counts().sort_index().tolist())

### Reading the output (Section 1, Example A)
- Both criteria point at **k=3**: the elbow flattens after 3, and silhouette peaks there (~0.70, comfortably high) — the algorithm recovered the three personas we planted.
- The profiling table is the deliverable: one cluster reads as low-recency/high-frequency/high-spend (*Champions*), one the opposite (*Dormant*), one in between. Cluster **numbers are arbitrary** (cluster 0 has no meaning); the *profile* is what you'd present.
- Real RFM data is messier — personas overlap and silhouette values of 0.2–0.4 are common and still useful. Also remember K-Means' blind spots: elongated, unequal-density, or nested clusters break it (Section 3 shows exactly that).

In [ ]:
# ============================================================
# Example B: bootstrap stability — the unsupervised 'validation set'
# ============================================================
# The question: is the k=3 structure REAL, or an artifact of this exact sample?
# The test: resample the customers with replacement 20 times, re-cluster each
# resample, and use each refit model to label the ORIGINAL points. If the
# structure is real, every refit should agree with the reference solution
# (ARI near 1). Reshuffling solutions = you were clustering noise.
# (Reuses X_scaled from Example A.)
from sklearn.cluster import KMeans

rng = np.random.default_rng(RANDOM_SEED)
n_boot = 20

print("k | stability ARI: mean  (min over 20 resamples)")
for k in [2, 3, 4, 5]:
    # Reference solution on the full data
    ref = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit(X_scaled)
    aris = []
    for b in range(n_boot):
        idx = rng.integers(0, len(X_scaled), len(X_scaled))    # bootstrap resample
        km_b = KMeans(n_clusters=k, n_init=10, random_state=b).fit(X_scaled[idx])
        # Label the ORIGINAL points with the resample-trained model, compare
        # to the reference labels. ARI ignores arbitrary cluster numbering.
        aris.append(adjusted_rand_score(ref.labels_, km_b.predict(X_scaled)))
    print(f"{k} | {np.mean(aris):.4f}  ({np.min(aris):.4f})")

print("\nk=3 is essentially perfectly stable. k=4 is the tell: mean drops AND the")
print("minimum craters — the 4th cluster is carved differently on every resample")
print("because there is no real 4th group to find. An unstable solution is the")
print("data telling you that granularity does not exist.")

### Reading the output (Section 1, Example B)
- **k=3 is near-perfectly stable** (ARI ≈ 0.999 across all resamples) — the three personas are structure, not luck. **k=4's minimum ARI craters** (~0.72): the phantom 4th cluster lands somewhere different on every resample. Instability is what "no real cluster there" looks like quantitatively.
- Subtle and important: **k=2 is also perfectly stable** — merging Regulars+Champions vs Dormant is *also* a real (coarser) structure. Stability tells you a partition is *reliable*, not that it's the *best* granularity — that's why you triangulate: silhouette peaked at 3, stability confirms 3 is solid, and business interpretability breaks the k=2 vs k=3 tie.
- This resample-and-compare pattern (via ARI between clusterings) works for *any* clustering algorithm, and it's cheap. Make it a habit before presenting any segmentation.

### 🧠 Can you answer? (Section 1 — K-Means)
1. **Sketch the proof that Lloyd's algorithm always terminates:** what quantity strictly decreases at both the assignment step and the update step, and why does a finite number of possible partitions finish the argument? Why does this guarantee only a *local* optimum?
2. **Prove that inertia is non-increasing in k** (hint: exhibit a (k+1)-cluster solution at least as good as any k-cluster solution). What does this imply about ever "minimizing" inertia directly?
3. **What does k-means++ initialization actually do, and what guarantee does it buy** (expected O(log k)-competitive inertia)? Why do we still keep `n_init=10` on top of it?
4. **Example B showed k=2 and k=3 both perfectly stable, but silhouette clearly preferred 3. Precisely characterize what stability measures vs what silhouette measures, and construct a dataset where stability is high for a k that silhouette rightly rejects.**
5. **Derive K-Means as a limiting case of a Gaussian Mixture Model:** which covariance structure, weight constraint, and what limit of the variance turns soft responsibilities into hard nearest-centroid assignment?
6. **Standardization forces all three RFM features to matter equally. Argue when that's the wrong choice** — e.g., the business insists Monetary should dominate — and show two mechanically different ways to encode feature importance into K-Means.
7. **A one-hot-encoded 'Region' column (20 categories) is added and everything is standardized. Explain at least two distinct things that go wrong for K-Means, and what k-modes / k-prototypes change.**
8. **In production you assign new customers to segments with `km.predict()` for a year without refitting. What quietly degrades, how would you detect it (name a concrete monitored statistic), and what's the trade-off in re-clustering monthly?**
9. **Why does K-Means become nearly useless in very high dimensions even after scaling — connect to the distance-concentration argument from the Supervised notebook's KNN section, and name the standard two-stage remedy.**

## 2. Hierarchical Clustering — Dendrogram Analysis
**Dataset Target:** Wine Dataset (Natural Multi-Group Chemistry Profiles)

### The concept, in plain words
**Agglomerative** (bottom-up) hierarchical clustering starts with every point as its own cluster, then repeatedly **merges the two closest clusters** until only one remains. The full merge history is drawn as a **dendrogram** — a tree where the height of each junction is the distance at which two clusters merged. Slice the tree horizontally at any height and the number of branches you cut = your number of clusters. Unlike K-Means, you don't commit to k up front; you choose it *after* seeing the structure.

### The choice that changes everything: linkage
"Distance between two clusters" needs a definition:
- **ward** — merge the pair that least increases within-cluster variance (tends to give compact, similar-size clusters; usually the best default).
- **complete** — distance between clusters = their *farthest* pair of points (compact but outlier-sensitive).
- **average** — mean of all cross-pair distances (a compromise; can produce straggly chains on some data).
- **single** — distance = the *closest* pair of points. Deliberately excluded from Example A and given its own demolition in **Example B**: it has a spectacular failure mode (chaining) that every practitioner must see once.

The code compares linkages on the same data — silhouette (internal) and ARI vs. the true cultivars (external, study-only) reveal how much linkage matters.

### Study Checklist
- [ ] Agglomerative Bottom-Up Merging Mechanics
- [ ] Linkage Criteria Comparison (ward vs. complete vs. average)
- [ ] Reading a Dendrogram — Cutting Height = Number of Clusters
- [ ] Cutting by `distance_threshold` Instead of n_clusters (Example B)
- [ ] Single-Linkage Chaining Failure (Example B)
- [ ] External Validation Against Known Labels (ARI, learning-only)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: Agglomerative Clustering + Dendrogram
# ============================================================
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import load_wine
from scipy.cluster.hierarchy import dendrogram, linkage   # scipy draws dendrograms

# 178 wines x 13 chemical features. There ARE 3 true cultivars (grape types),
# but we hide them from the algorithm and use them only to grade it afterwards.
wine = load_wine()
X_scaled = StandardScaler().fit_transform(wine.data)   # distance-based -> scale first

# --- Step 1: same data, three linkage definitions ---
print("linkage  | silhouette (internal) | ARI vs true cultivars (external)")
for link in ['ward', 'complete', 'average']:
    labels = AgglomerativeClustering(n_clusters=3, linkage=link).fit_predict(X_scaled)
    print(f"{link:<8} | {silhouette_score(X_scaled, labels):.4f}                "
          f" | {adjusted_rand_score(wine.target, labels):.4f}")

# --- Step 2: the dendrogram — the algorithm's full merge history ---
# linkage() (scipy) computes the merge tree; dendrogram() draws it.
# truncate_mode='lastp' shows only the last 25 merges so the plot stays readable.
# HOW TO READ IT: y-axis = merge distance. Long vertical stems = well-separated
# groups. A horizontal cut across 3 stems = a 3-cluster solution.
plt.figure(figsize=(11, 4))
dendrogram(linkage(X_scaled, method='ward'), truncate_mode='lastp', p=25)
plt.title('Ward Linkage Dendrogram (last 25 merges)')
plt.xlabel('Sample index / (size of merged cluster)')
plt.ylabel('Merge distance')
plt.tight_layout(); plt.show()

### Reading the output (Section 2, Example A)
- **Linkage is not a detail**: ward reaches ARI ≈ 0.79 (strong agreement with the true cultivars) while average linkage collapses to ARI ≈ 0 — no better than random — *on identical data*. Algorithm choices inside the "same" method can swing results from excellent to useless.
- Note the silhouette values are modest (~0.28) even for ward, yet ARI is high. Silhouette measures geometric tightness in 13-D; clusters can be correct without being geometrically snug. No single validation number tells the whole story.
- In the dendrogram, the two or three **tallest vertical stems** near the top mark the natural major divisions — cutting just below them yields the 3-group solution. That visual, exploratory choice of k is hierarchical clustering's main gift over K-Means. (Its cost: O(n²) memory — impractical beyond ~tens of thousands of rows.)

In [ ]:
# ============================================================
# Example B: cutting by height + single linkage's chaining disaster
# ============================================================
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs

# --- Part 1: cut the tree by DISTANCE, not by cluster count ---
# Example A's dendrogram showed the last merges happen around heights ~28 and
# ~35. Cutting below the tall stems (threshold 25) should yield 3 clusters —
# the tree, not us, decides how many groups exist at that granularity.
# (Reuses X_scaled from Example A.)
for t in [20, 25, 30]:
    ac = AgglomerativeClustering(n_clusters=None, distance_threshold=t,
                                 linkage='ward').fit(X_scaled)
    print(f"distance_threshold={t}: -> {ac.n_clusters_} clusters")
print("Threshold-cutting is often better in production: as data grows, 'clusters")
print("tighter than distance d' stays meaningful while a fixed k may not.\n")

# --- Part 2: the chaining failure ---
# Two clean, well-separated blobs (200 points each)... plus a THIN BRIDGE of
# just 15 points between them. Single linkage measures cluster distance by the
# single CLOSEST pair — so it happily walks across the bridge, stepping-stone
# by stepping-stone, and fuses the blobs into one giant chain.
rng = np.random.default_rng(RANDOM_SEED)
X_blobs, y_blobs = make_blobs(n_samples=[200, 200], centers=[[-4, 0], [4, 0]],
                              cluster_std=0.7, random_state=RANDOM_SEED)
bridge = np.column_stack([np.linspace(-3.2, 3.2, 15), rng.normal(0, 0.08, 15)])
X_br = np.vstack([X_blobs, bridge])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, link in zip(ax, ['single', 'ward']):
    labels = AgglomerativeClustering(n_clusters=2, linkage=link).fit_predict(X_br)
    # grade only on the 400 genuine blob points (the bridge has no true label)
    ari = adjusted_rand_score(y_blobs, labels[:400])
    a.scatter(*X_br.T, c=labels, s=8, cmap='coolwarm')
    a.set_title(f"{link} linkage — ARI {ari:.2f}, sizes {np.bincount(labels).tolist()}")
plt.tight_layout(); plt.show()

print("15 bridge points defeated 400 blob points under single linkage: one")
print("cluster holds ~414 points, the 'other' is a single stray point (ARI 0.0).")
print("Ward, which merges by variance increase rather than closest pair, cuts")
print("the bridge and recovers both blobs perfectly (ARI 1.0).")
print("Flip side: on genuinely elongated/manifold-shaped clusters (Section 3's")
print("moons), chaining is exactly what you WANT — single linkage is DBSCAN's")
print("close cousin. Failure mode and superpower are the same mechanism.")

### Reading the output (Section 2, Example B)
- The threshold sweep shows the tree answering "how many clusters?" itself: cut at 20 or 25 → 3 clusters; cut at 30 (above one tall stem) → 2. In production pipelines, a distance threshold ("merge anything tighter than d") is often more durable than a hard-coded k as data volume grows.
- The chaining demo is the sharpest algorithm-assumption lesson in this notebook: **15 low-density stepping stones** let single linkage fuse two blobs of 200 points each — the "2-cluster" solution is 414-points-vs-1-stray, ARI 0.0. Ward is immune because a merge across the bridge would hugely increase variance.
- And the twist worth remembering: chaining isn't a bug in all worlds. On elongated, manifold-like structure it's the *correct* behavior — which is why single linkage relates closely to DBSCAN and minimum-spanning-tree clustering. There are no good or bad linkages, only matched or mismatched geometry.

### 🧠 Can you answer? (Section 2 — Hierarchical Clustering)
1. **Why is naive agglomerative clustering O(n²) in memory and O(n³) in time (what is recomputed at every merge), and what do the Lance-Williams update formulas reduce it to? What does this cap practical dataset sizes at?**
2. **Ward's criterion: write in words what quantity a ward merge minimizes the increase of, and show its deep kinship with K-Means' objective. Why does ward (in sklearn/scipy) require Euclidean distance while average/complete accept any metric?**
3. **Explain the chaining mechanism precisely: after the first bridge point joins a blob's cluster, what does single linkage's cluster-to-cluster distance become for the next bridge point? Why does the bridge's *density* not matter at all to single linkage?**
4. **Define cophenetic distance between two points, and what the cophenetic correlation coefficient measures. What would a low value (~0.4) tell you about trusting the dendrogram's story?**
5. **Wine gave silhouette 0.28 but ARI 0.79. Reconcile these numbers: what geometric property can 13-D data have that keeps silhouette low even when the partition is essentially correct?**
6. **Dendrogram inversions: under which linkage methods (hint: centroid, median) can a later merge happen at a LOWER height than an earlier one, why is that geometrically possible, and why are single/complete/average/ward guaranteed monotone?**
7. **`AgglomerativeClustering` has no `.predict()` for new points. Explain why that's structural rather than an oversight, and evaluate two workarounds (nearest-centroid assignment; train a classifier on the cluster labels) — what does each silently assume?**
8. **Why is divisive (top-down) hierarchical clustering almost never used exactly — what is the size of the first split's search space — and what heuristic (e.g., bisecting K-Means) makes a practical version?**
9. **You must cluster 5 million customer embeddings hierarchically for a taxonomy. Design the actual pipeline (hint: no O(n²) allowed) — what do you pre-reduce with, what do you subsample or mini-batch, and how do you validate the result at that scale?**

## 3. DBSCAN — Density Clusters & Noise
**Dataset Target:** Non-Convex Synthetic Shapes (Where K-Means Fails)

### The concept, in plain words
DBSCAN defines clusters by **density**, not by distance to a center. Two parameters: `eps` (the radius of each point's neighborhood) and `min_samples` (how many neighbors make a neighborhood "dense"). Then:
- **Core point** — has ≥ `min_samples` neighbors within `eps`.
- **Border point** — within `eps` of a core point but not dense itself.
- **Noise** — neither. Labeled **−1** and belonging to *no* cluster.

Clusters grow by chaining core points whose neighborhoods overlap — so a cluster can be *any shape* a dense path can trace: crescents, rings, spirals.

### Why this section exists
K-Means can only produce **convex** regions (each point goes to its nearest centroid, carving space into flat-walled cells). On two interleaved crescent moons it fails structurally — no amount of tuning fixes it. DBSCAN handles them natively, plus gives you noise detection and no need to choose k. The price: `eps` is the new hard choice — and v2 only *named* the principled way to pick it. **Example B implements it (the k-distance elbow), then shows the failure mode no eps can fix: clusters of different densities.**

### Study Checklist
- [ ] Density Reachability (core / border / noise points)
- [ ] eps & min_samples Sensitivity Sweep
- [ ] The k-Distance Elbow Heuristic for Choosing eps (Example B)
- [ ] Noise Label (−1) Handling in Downstream Analysis
- [ ] Non-Convex Cluster Recovery (moons) — K-Means Failure Contrast
- [ ] Varying-Density Failure Mode — Why One Global eps Can't Fit Two Densities (Example B)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: DBSCAN on Non-Convex Data
# ============================================================
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons

# make_moons: two interleaved crescents — the canonical K-Means-breaker.
# y_true (which crescent each point belongs to) is kept ONLY for grading.
X_moons, y_true = make_moons(n_samples=600, noise=0.08, random_state=RANDOM_SEED)
X_scaled = StandardScaler().fit_transform(X_moons)

# --- Step 1: demonstrate the K-Means failure ---
# K-Means partitions space into convex cells around centroids; a crescent
# wrapping around another crescent cannot fit in a convex cell.
km_labels = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_SEED).fit_predict(X_scaled)
print(f"K-Means ARI on moons: {adjusted_rand_score(y_true, km_labels):.4f}"
      "  <- structurally poor, no tuning can fix this")

# --- Step 2: eps sensitivity sweep ---
# Too small -> everything is 'noise' / fragments. Too large -> neighborhoods
# bridge the gap between crescents and merge everything into one blob.
print("\neps  | clusters found | noise points | ARI")
for eps in [0.1, 0.2, 0.3, 0.5]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_scaled)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)  # -1 isn't a cluster
    n_noise = (db.labels_ == -1).sum()
    print(f"{eps:.1f} | {n_clusters:>14} | {n_noise:>12} | {adjusted_rand_score(y_true, db.labels_):.4f}")

# --- Step 3: see it ---
best = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_scaled)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(*X_scaled.T, c=km_labels, s=8, cmap='coolwarm')
ax[0].set_title('K-Means: slices straight across (wrong)')
ax[1].scatter(*X_scaled.T, c=best, s=8, cmap='coolwarm')
ax[1].set_title('DBSCAN eps=0.2: follows the crescents')
plt.tight_layout(); plt.show()

### Reading the output (Section 3, Example A)
- The sweep is the lesson: eps=0.1 shatters the data into ~27 fragments with heavy noise; **eps=0.2 nails it** (2 clusters, ARI ≈ 0.97); eps≥0.3 bridges the gap and merges everything into one cluster (ARI = 0, and note silhouette-style metrics can't even warn you here). DBSCAN's quality cliff around eps is steep — always sweep, never guess.
- The plots make K-Means' failure visceral: it draws a straight frontier through both crescents because that's all convex cells can do. Matching **algorithm assumptions to data shape** beats tuning every time.
- Practical note on the −1 label: noise points are *unassigned*, not "cluster −1". Downstream code that does `groupby(labels)` must handle them explicitly — a very common silent bug.

In [ ]:
# ============================================================
# Example B: choosing eps like an adult — then the wall no eps can climb
# ============================================================
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

# --- Part 1: the k-distance elbow, implemented ---
# For every point, compute the distance to its (min_samples)th nearest
# neighbor, sort ascending, and plot. Points inside dense regions have SMALL
# k-distances; noise/sparse points have LARGE ones. The knee of the curve is
# the natural boundary — a principled starting eps.
# (Reuses X_scaled = scaled moons from Example A.)
k = 5   # match min_samples
dists, _ = NearestNeighbors(n_neighbors=k).fit(X_scaled).kneighbors(X_scaled)
kdist = np.sort(dists[:, -1])                 # each point's distance to 5th neighbor

plt.figure(figsize=(8, 3.5))
plt.plot(kdist)
plt.axhline(0.2, color='red', ls='--', lw=1, label='eps=0.2 (Example A winner)')
plt.legend()
plt.title(f'k-distance plot (distance to {k}th neighbor, sorted)')
plt.xlabel('points, sorted'); plt.ylabel(f'{k}th-NN distance')
plt.tight_layout(); plt.show()
print(f"90% of points have a 5th-NN distance below {np.quantile(kdist, 0.9):.3f};")
print("the curve turns sharply upward right around 0.15-0.25 — the knee lands")
print("on the same eps=0.2 the brute-force sweep found. Heuristic validated.\n")

# --- Part 2: the failure no eps fixes — two clusters, two densities ---
# A tight blob (std 0.25) and a diffuse blob (std 1.8). The dense cluster
# needs a SMALL eps; the sparse one needs a LARGE eps. One global eps must
# betray one of them.
rng = np.random.default_rng(RANDOM_SEED)
dense  = rng.normal([0, 0], 0.25, (300, 2))
sparse = rng.normal([5, 0], 1.8,  (300, 2))
X_dd = np.vstack([dense, sparse])
y_dd = np.array([0]*300 + [1]*300)

print("eps  | clusters | noise | ARI    | what happened")
stories = {0.2: 'dense cluster perfect; sparse one mostly labeled NOISE',
           0.5: 'sparse cluster shatters into fragments',
           0.9: 'neighborhoods bridge the gap -> one giant blob',
           1.5: 'one blob, of course'}
for eps in [0.2, 0.5, 0.9, 1.5]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_dd)
    ncl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    noise = int((db.labels_ == -1).sum())
    print(f"{eps:<4} | {ncl:>8} | {noise:>5} | {adjusted_rand_score(y_dd, db.labels_):.4f} | {stories[eps]}")

print("\nNo row is right: at eps=0.2 HALF the dataset (the sparse cluster) is")
print("'noise'; by eps=0.9 everything has merged. A single global density")
print("threshold cannot describe a world with two densities. This exact gap is")
print("why HDBSCAN exists — it builds a hierarchy over all eps values at once")
print("and extracts the most stable clusters per branch. Know the tool AND the")
print("moment it's needed.")

### Reading the output (Section 3, Example B)
- The k-distance knee lands right where the brute-force sweep put eps — the heuristic works, and unlike the sweep it doesn't need ground-truth ARI to find the answer. On real (label-free) data, the knee is your *only* principled guide, so practice reading it here where you can verify it.
- The varying-density table is the honest wall: at the eps that respects the dense cluster, **288 of 600 points — nearly the whole sparse cluster — are branded noise**; raise eps enough to embrace the sparse cluster and the two clusters merge first. DBSCAN's single global eps assumes *one* notion of "dense" for the whole dataset.
- HDBSCAN (in sklearn ≥1.3 as `sklearn.cluster.HDBSCAN`) resolves exactly this by exploring all densities hierarchically — a one-line swap worth trying on this data as an exercise.

### 🧠 Can you answer? (Section 3 — DBSCAN)
1. **Give the precise definitions of directly density-reachable, density-reachable, and density-connected — and explain why a DBSCAN cluster is defined as a maximal set of density-*connected* points rather than density-*reachable* ones (asymmetry!).**
2. **DBSCAN is often called deterministic, yet border points can legitimately end up in different clusters depending on processing order. Explain exactly when that happens and why core points never suffer this ambiguity.**
3. **Justify the k-distance heuristic from first principles: why should the curve have a knee at all, what do the two regimes (flat left, steep right) correspond to, and why should k be tied to min_samples?**
4. **What does raising min_samples from 5 to 25 do to (a) the effective density threshold, (b) noise volume, (c) robustness to bridge points like Section 2's? Where does the common min_samples ≈ 2·dims rule of thumb come from?**
5. **In Example B, eps=0.2 scored ARI 0.89 despite labeling 288 points as noise. Dissect how ARI treats the noise label here, and argue whether ARI flattered or punished DBSCAN — what would you report instead for a fair account?**
6. **Two reasons silhouette score is a poor referee for DBSCAN output: what does it do with the −1 points, and why does its compactness logic systematically punish correct non-convex clusters? What is DBCV designed to fix?**
7. **DBSCAN with an R*-tree index runs ~O(n log n). In which regime does the index degrade the whole thing back to O(n²), and which earlier section's curse explains why?**
8. **Explain mechanically why HDBSCAN survives the two-density dataset: what is mutual reachability distance, what hierarchy is built, and what does "stability" mean when it extracts flat clusters from the tree?**
9. **Design the production assignment rule for a new streaming transaction given a fitted DBSCAN (which has no `.predict()`): state the rule, its edge cases (near two clusters; near only border points), and when accumulated drift forces a re-fit.**

## 4. Gaussian Mixture Models — Soft Clustering & BIC
**Dataset Target:** Overlapping Synthetic Gaussian Populations

### The concept, in plain words
A GMM assumes the data was generated by **n overlapping Gaussian (bell-curve) blobs**, and estimates each blob's center, shape (covariance), and weight. Fitting uses the **EM algorithm**, an alternation directly analogous to K-Means' two steps:
- **E-step:** given current blobs, compute each point's *responsibility* — the probability it came from each blob.
- **M-step:** given responsibilities, re-estimate each blob's parameters as weighted averages.

Two upgrades over K-Means fall out of this:
1. **Soft assignments.** Every point gets a *probability per cluster* (`predict_proba`), not a hard label. A point that's 55/45 between two segments is flagged as genuinely ambiguous rather than forced into a box — often exactly what a business wants to know.
2. **Cluster shape.** With `covariance_type='full'`, blobs can be stretched ellipses of different sizes and orientations. (With `'spherical'` and equal weights, GMM essentially *becomes* K-Means — a nice way to see K-Means as a special case.) **Example B puts a price tag on this choice** by comparing covariance types on deliberately stretched clusters — and then uses the fitted GMM as a *generator* of new synthetic data, the ability that makes it a "generative model."

### Choosing n_components without labels: BIC
The **Bayesian Information Criterion** = model fit (log-likelihood) *minus* a penalty for parameter count. More components always fit better, but BIC charges rent for them; **the minimum BIC** marks the sweet spot. This gives a *principled* answer to "how many clusters?" — arguably cleaner than eyeballing an elbow.

### Study Checklist
- [ ] EM Algorithm Intuition (E-step responsibilities, M-step parameter updates)
- [ ] Soft Assignments — predict_proba per cluster
- [ ] Covariance Types Priced by BIC on Stretched Data (Example B)
- [ ] Model Selection via BIC/AIC (principled choice of n_components)
- [ ] GMM as a Generative Model — sampling synthetic data (Example B)
- [ ] When GMM Beats K-Means (elliptical, overlapping clusters)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: GMM with BIC Model Selection
# ============================================================
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

# Four blobs with DIFFERENT spreads (cluster_std varies) — the kind of
# unequal, overlapping structure where GMM's flexible covariances shine.
# The true count (4) is hidden from the model; BIC must find it.
X_blob, _ = make_blobs(n_samples=900, centers=4,
                       cluster_std=[1.0, 2.5, 0.7, 1.8],
                       random_state=RANDOM_SEED)
X_scaled = StandardScaler().fit_transform(X_blob)

# --- Step 1: BIC sweep over candidate component counts ---
# BIC = -2*log-likelihood + penalty*(number of parameters). LOWER is better.
# AIC is similar with a lighter penalty (tends to pick more components).
print("n_components | BIC        | AIC")
bics = []
for n in range(1, 8):
    gmm = GaussianMixture(n_components=n,
                          covariance_type='full',   # each blob: own ellipse shape
                          random_state=RANDOM_SEED).fit(X_scaled)
    bics.append(gmm.bic(X_scaled))
    print(f"{n:>12} | {gmm.bic(X_scaled):>10.1f} | {gmm.aic(X_scaled):>10.1f}")

best_n = int(np.argmin(bics)) + 1     # +1 because the sweep started at n=1
print(f"\nBIC minimum at n_components = {best_n}  (true answer: 4)")

# --- Step 2: soft assignments — the defining difference vs K-Means ---
gmm = GaussianMixture(n_components=best_n, covariance_type='full',
                      random_state=RANDOM_SEED).fit(X_scaled)
proba = gmm.predict_proba(X_scaled)   # shape: (n_points, n_components)

# Points whose strongest membership is <70% sit in genuine overlap zones —
# information a hard-labeling algorithm simply throws away.
ambiguous = (proba.max(axis=1) < 0.7).sum()
print(f"Points with max membership < 70%: {ambiguous} of {len(X_scaled)}")
print("Example row of membership probabilities:", proba[0].round(3))

### Reading the output (Section 4, Example A)
- **BIC bottoms out exactly at 4** — the true blob count we hid — then rises as the penalty outweighs marginal fit. Watch AIC: with its lighter penalty it often stays flat or keeps drifting down; BIC's stronger penalty makes it the more conservative (and usually safer) chooser.
- The example membership row shows what "soft" means: probabilities across components summing to 1. Only a handful of points fall below 70% confidence here because the blobs are well-separated after scaling; on real customer data expect far more — and treat those ambiguous points as a *finding* (hybrid customers), not an annoyance.
- Caveat for honesty: GMM assumes Gaussian blobs. On the crescent moons of Section 3, it fails just like K-Means — soft labels don't fix wrong shape assumptions. Every model in this notebook has a geometry it believes in; your job is matching it to the data's.

In [ ]:
# ============================================================
# Example B: pricing the covariance assumption + GMM as a data generator
# ============================================================
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

# --- Part 1: covariance types on STRETCHED clusters ---
# Shear the blobs with a linear transform so every cluster becomes a tilted
# ellipse. Now the covariance_type choice has consequences:
#   spherical -> every blob must be a circle       (cheapest, most wrong here)
#   diag      -> axis-aligned ellipses only         (middle)
#   full      -> arbitrary tilted ellipses          (priciest, matches reality)
# BIC judges the fit-vs-parameter-count trade for us.
X_raw, _ = make_blobs(n_samples=900, centers=4, cluster_std=[1.0, 2.5, 0.7, 1.8],
                      random_state=RANDOM_SEED)
shear = np.array([[0.6, -0.6],
                  [-0.3, 0.9]])
X_stretch = StandardScaler().fit_transform(X_raw @ shear)   # tilted ellipses now

print("covariance_type | BIC (lower=better) | # parameters")
for ct in ['spherical', 'diag', 'full']:
    g = GaussianMixture(n_components=4, covariance_type=ct,
                        random_state=RANDOM_SEED).fit(X_stretch)
    print(f"{ct:<15} | {g.bic(X_stretch):>18.1f} | {g._n_parameters():>12}")
print("-> 'full' wins by a landslide DESPITE the biggest parameter penalty:")
print("   when the data really is tilted ellipses, paying for covariance is a")
print("   bargain. On round blobs the ranking flips — assumptions have prices.\n")

# --- Part 2: the 'generative' in generative model ---
# A fitted GMM is a full probability distribution over the feature space, so
# it can SAMPLE new, never-observed points. Discriminative clusterers
# (K-Means, DBSCAN) fundamentally cannot do this.
gmm = GaussianMixture(n_components=4, covariance_type='full',
                      random_state=RANDOM_SEED).fit(X_stretch)
X_fake, comp = gmm.sample(300)

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
ax[0].scatter(*X_stretch.T, s=6, alpha=0.5)
ax[0].set_title('Real data (sheared blobs)')
ax[1].scatter(*X_fake.T, c=comp, s=6, cmap='tab10', alpha=0.7)
ax[1].set_title('300 points SAMPLED from the fitted GMM')
plt.tight_layout(); plt.show()
print("Sampled points per component:", np.bincount(comp).tolist())
print("Uses: simulation, augmentation, density-based anomaly scores")
print("(gmm.score_samples: low log-likelihood = 'this point is unlike any blob'")
print("— an alternative to Section 7's Isolation Forest). The caveat: samples")
print("are only as realistic as the Gaussian assumption that generated them.")

### Reading the output (Section 4, Example B)
- BIC prices the assumption cleanly: on sheared data, `full` (23 parameters) beats `diag` (19) and `spherical` (15) by over a thousand BIC points — the extra covariance parameters pay for themselves many times over. Run the same comparison on round blobs and the ranking flips. **Covariance type is a statement about cluster geometry, and BIC is how you test the statement.**
- The right panel is the capability the word "generative" promises: the fitted model *is* a probability distribution, so it can mint new plausible data. K-Means can tell you where a point belongs; it cannot tell you what the data looks like.
- `score_samples` (per-point log-likelihood) quietly gives you a second anomaly detector for free — points that no component explains well get very negative scores. Keep this in your pocket for Section 7.

### 🧠 Can you answer? (Section 4 — Gaussian Mixture Models)
1. **Sketch why each EM iteration can never decrease the observed-data log-likelihood** (the E-step builds a tight lower bound; the M-step maximizes it). Why does this guarantee convergence but NOT to a global optimum — and what practical parameter (`n_init`) is the confession?
2. **Make the K-Means connection exact: with spherical covariances of shared variance σ², equal weights, show what the responsibility of component j for point x converges to as σ² → 0.**
3. **Count the parameters for k components in d dimensions under spherical, diag, full, and tied covariances (means + covariances + weights). Verify your formula against the printed `_n_parameters()` for d=2, k=4.**
4. **Maximum-likelihood GMM is technically ill-posed: describe the singularity where one component's likelihood diverges to infinity, and how `reg_covar` (and a Bayesian prior) tames it.**
5. **Write the AIC and BIC penalty terms. Which criterion is *consistent* (selects the true model as n→∞, if it's in the candidate set) and which is *efficient*? Why does AIC's lighter penalty make it drift toward more components in Example A?**
6. **Label switching: why is the likelihood invariant to permuting component indices, and what concrete problems does that cause when comparing two fitted GMMs or averaging parameters across runs?**
7. **On the moons data, a 2-component GMM's `predict_proba` will produce confident-looking probabilities that are meaningless. Explain why responsibilities are only as calibrated as the model is well-specified, and how you would detect the misspecification without labels.**
8. **For d=50 features and k=10 components with full covariance: how many parameters, roughly how many rows would you insist on, and which covariance compromise (or preprocessing step) would you reach for first?**
9. **You need to cluster AND flag anomalies AND simulate synthetic customers for a stress test. Argue why one fitted GMM can serve all three jobs, and state the single assumption all three outputs inherit.**

## 5. PCA — Dimensionality Reduction & Variance
**Dataset Target:** Breast Cancer Wisconsin (30 Correlated Features)

### The concept, in plain words
**Principal Component Analysis** finds new axes for your data. PC1 is the direction along which the data varies *most*; PC2 the most-varying direction perpendicular to PC1; and so on. Each principal component is a weighted mix (a **linear combination**) of the original features. Because early components hoard the variance, you can often keep a handful of them and discard the rest — compressing 30 correlated features into ~10 nearly-lossless ones.

Two ways to think about it:
- *Statistics view:* eigendecomposition of the covariance matrix; eigenvalues = variance per component.
- *Geometry view:* rotate the cloud of points so its longest axis lines up with axis 1, second-longest with axis 2, ... then drop the short axes.

**A third view v3 adds (Example B):** PCA is a lossy compressor with an `inverse_transform` — project down, reconstruct back, and *look at what survived*. On images, you can literally watch information return as you keep more components.

### Why scale FIRST (non-negotiable)
PCA chases variance. In raw units, a feature measured in thousands has enormous variance *purely from its units* and would hijack PC1. Standardizing makes "variance" mean *information*, not *unit size*. (Fine print for Example B: when all features share the same units and scale — like pixel intensities — working unscaled is legitimate and keeps reconstructions viewable. Scaling is about comparability, not ritual.)

### Choosing how many components
- **95% cumulative variance rule** — keep the smallest set of components explaining 95% of total variance (the scree curve below shows where that lands).
- **Loadings** — each component's weights over original features. Reading PC1's largest loadings tells you *what the dominant axis actually measures* — turning math back into meaning.

### Study Checklist
- [ ] Covariance Matrix Eigendecomposition Intuition
- [ ] Scaling BEFORE PCA (variance in raw units is meaningless across features)
- [ ] Explained Variance Ratio & Cumulative Scree Curve
- [ ] Choosing Components (95% variance rule vs. scree elbow)
- [ ] Loadings — Which Original Features Drive Each Component
- [ ] Reconstruction: PCA as a Lossy Compressor (Example B — inverse_transform)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: PCA — Variance Analysis & Projection
# ============================================================
from sklearn.decomposition import PCA
from sklearn.datasets import load_breast_cancer

# 569 tumors x 30 features. The 30 features are HIGHLY correlated
# (radius, perimeter, area all measure size) — ideal PCA territory:
# lots of redundancy to squeeze out.
cancer = load_breast_cancer()
X_scaled = StandardScaler().fit_transform(cancer.data)   # scale FIRST — always

# --- Step 1: fit full PCA and examine the variance ledger ---
pca = PCA().fit(X_scaled)                                # keep all 30 components
cumvar = np.cumsum(pca.explained_variance_ratio_)        # running total of variance
n95 = int(np.searchsorted(cumvar, 0.95)) + 1             # components to reach 95%
print(f"PC1 alone explains {pca.explained_variance_ratio_[0]:.1%} of all variance")
print(f"Components needed for 95%: {n95} of {X_scaled.shape[1]} -> "
      f"{X_scaled.shape[1]-n95} dimensions were mostly redundancy")

# --- Step 2: scree curve + 2D projection ---
# The 2D scatter is colored by the true diagnosis, which PCA NEVER SAW —
# if colors separate along PC1, the unsupervised axes align with the real signal.
X_2d = PCA(n_components=2).fit_transform(X_scaled)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(range(1, len(cumvar) + 1), cumvar, marker='o', ms=3)
ax[0].axhline(0.95, color='red', ls='--', lw=1)
ax[0].set(title='Cumulative Explained Variance (scree)', xlabel='# components',
          ylabel='cumulative ratio')
ax[1].scatter(X_2d[:, 0], X_2d[:, 1], c=cancer.target, s=6, cmap='coolwarm', alpha=0.6)
ax[1].set(title='2D projection (color = diagnosis, unseen by PCA)', xlabel='PC1', ylabel='PC2')
plt.tight_layout(); plt.show()

# --- Step 3: loadings — what does PC1 actually measure? ---
# components_[0] holds PC1's weight on each original feature.
loadings = pd.Series(pca.components_[0], index=cancer.feature_names)
print("\nLargest |loadings| on PC1 (the features defining the dominant axis):")
print(loadings.abs().sort_values(ascending=False).head(5).round(3))

### Reading the output (Section 5, Example A)
- **10 of 30 components carry 95% of the variance** — two-thirds of the measured dimensions were largely restating the other third. This is typical of hand-designed feature sets, where many features measure the same underlying thing.
- The 2D scatter shows malignant and benign tumors separating substantially **along PC1 alone** — remarkable, because PCA never saw the diagnosis. When unsupervised structure aligns with a label like this, the features contain strong signal (and a downstream classifier will have an easy time — exactly why PCA+SVM works so well in the Supervised notebook's Section 7).
- PC1's top loadings are all concavity/size measures — so PC1 ≈ *"overall tumor severity axis."* Naming your components via loadings is what makes PCA a communication tool, not just compression. One caution: PCA maximizes variance, not class separation — the alignment here is a happy property of this data, never a guarantee.

In [ ]:
# ============================================================
# Example B: PCA as a compressor you can SEE — digit reconstruction
# ============================================================
# project 64-pixel digit images down to n components, inverse_transform back
# to 64 pixels, and look at what survived. Compression made visible.
# NOTE we deliberately work UNSCALED here: pixels already share one unit and
# scale (0-16), and skipping the scaler keeps reconstructions viewable as
# images. Scaling is for comparability across unlike features, not a ritual.
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

digits = load_digits()
X_img = digits.data                      # 1797 x 64, pixel intensities 0-16

print("components | variance kept | reconstruction MSE")
recons = {}
for nc in [5, 15, 40]:
    p = PCA(n_components=nc).fit(X_img)
    X_rec = p.inverse_transform(p.transform(X_img))    # 64 -> nc -> 64
    recons[nc] = X_rec
    mse = np.mean((X_img - X_rec) ** 2)
    print(f"{nc:>10} | {p.explained_variance_ratio_.sum():>13.1%} | {mse:.3f}")

# Show 6 sample digits: original vs reconstructions at each budget
rng = np.random.default_rng(RANDOM_SEED)
picks = rng.choice(len(X_img), 6, replace=False)
fig, axes = plt.subplots(4, 6, figsize=(9, 6.5))
for col, idx in enumerate(picks):
    axes[0, col].imshow(X_img[idx].reshape(8, 8), cmap='gray_r')
    axes[0, col].set_title(f"digit {digits.target[idx]}", fontsize=9)
    for row, nc in enumerate([5, 15, 40], start=1):
        axes[row, col].imshow(recons[nc][idx].reshape(8, 8), cmap='gray_r')
for row, label in zip(axes[:, 0], ['original', '5 comps', '15 comps', '40 comps']):
    row.set_ylabel(label, fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('PCA reconstruction: information returning as components increase')
plt.tight_layout(); plt.show()

### Reading the output (Section 5, Example B)
- At **5 components (≈55% variance)** digits are ghostly blurs — recognizable as "some digit" but often not *which*. At **15 (≈84%)** identity is clear. At **40 (≈99%)** reconstructions are visually indistinguishable from originals: the last 24 components were carrying almost nothing but pixel noise.
- This is the same 95%-variance story as Example A, but now your eyes are the metric — and the equivalence it demonstrates is the deep one: **PCA's max-variance directions are exactly the min-reconstruction-error directions.** Keeping the top components *is* optimal lossy linear compression.
- Practical bridge: "compress → reconstruct → measure error" is also an anomaly-detection pattern (a point that reconstructs badly doesn't live on the data's main subspace) and the conceptual ancestor of autoencoders — which are, at heart, non-linear PCA.

### 🧠 Can you answer? (Section 5 — PCA)
1. **Derive PC1: maximize $w^\top \Sigma w$ subject to $\|w\|=1$ via a Lagrange multiplier, and show the solution is the top eigenvector with the eigenvalue as the variance captured. Where does orthogonality of subsequent PCs come from?**
2. **State and sketch the equivalence Example B demonstrated: why is the variance-maximizing k-dimensional subspace the same one that minimizes total squared reconstruction error?**
3. **Show that standardizing before PCA is identical to running PCA on the correlation matrix instead of the covariance matrix. Then defend Example B's choice to skip scaling for pixel data — what property of the features makes raw covariance meaningful there?**
4. **Loadings' signs are arbitrary (a PC and its negation are the same axis). What concrete confusion does this cause when comparing PCA fits across two time periods or datasets, and how do practitioners stabilize interpretation?**
5. **Five features all measure tumor 'size' almost redundantly. Explain what happens to their individual loadings on PC1 and why interpreting a single loading in isolation ('perimeter matters most') can mislead — connect to the same correlated-feature trap as regression coefficients.**
6. **Construct (describe) a 2-class dataset where the discriminative direction is the LOWEST-variance direction, so PCA-then-classify at 95% variance destroys the signal. What supervised alternatives (LDA, PLS) change about the objective?**
7. **The 95% rule is arbitrary. Name and explain one more principled selector — parallel analysis (compare eigenvalues to those of shuffled data), cross-validated reconstruction error, or Minka's MLE (`n_components='mle'`) — and its failure modes.**
8. **When PCA feeds a supervised model, why must scaler AND PCA be fit inside each CV fold even though PCA never sees labels? State precisely what statistic leaks from the test fold if you don't. (Same question as Supervised S7 — can you answer it more sharply now?)**
9. **What does `whiten=True` do to the transformed components, which downstream algorithms want it and why, and what's the danger of whitening directions whose eigenvalues are near zero?**

## 6. t-SNE — Non-Linear Visualization
**Dataset Target:** Handwritten Digits (64-Dim → 2-Dim Embedding)

### The concept, in plain words
PCA is linear — it can only rotate and project. **t-SNE** (t-distributed Stochastic Neighbor Embedding) is non-linear: it computes, in the original high-dimensional space, a probability that each pair of points are "neighbors," then arranges points on a 2-D canvas so those neighbor probabilities are matched as well as possible. Result: points that were close in 64-D end up close in 2-D — even if the structure is curved or tangled in ways PCA cannot flatten.

### The fine print (this is where people go wrong)
- **Only local structure is preserved.** Within-cluster layout is meaningful; **distances *between* clusters and cluster *sizes* are NOT** — a cluster appearing "far away" or "bigger" means nothing.
- **Perplexity** (~5–50) sets the effective neighborhood size the algorithm tries to preserve. Different perplexities give visibly different maps — v2 said "always try a couple"; **Example B actually does it**, side by side, so you can see what survives and what's layout artifact.
- **Visualization-only.** t-SNE has no `.transform()` for new points and its axes have no meaning — never feed t-SNE coordinates into a downstream model as features.
- **PCA pre-reduction first** (64→30 here) is standard practice: it strips noise dimensions and cuts computation substantially.

### Study Checklist
- [ ] Neighbor-Probability Matching Intuition (local structure preservation)
- [ ] Perplexity Sensitivity — Demonstrated Side by Side (Example B)
- [ ] Why t-SNE Is Visualization-Only (inter-cluster distances/densities NOT meaningful)
- [ ] PCA Pre-Reduction for Speed on High-Dim Data
- [ ] Never Fit-Transform New Points (no out-of-sample transform)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: t-SNE Embedding of Digits
# ============================================================
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

# 1,797 handwritten digits, each a 64-pixel vector. Ten TRUE groups (digits
# 0-9) exist, but t-SNE never sees them — we color by them afterwards to
# grade the map by eye.
digits = load_digits()
X_scaled = StandardScaler().fit_transform(digits.data)

# --- Step 1: PCA pre-reduction, 64 -> 30 dims ---
# Keeps the signal-bearing directions, drops noise dims, speeds up t-SNE.
X_pca = PCA(n_components=30, random_state=RANDOM_SEED).fit_transform(X_scaled)

# --- Step 2: t-SNE down to 2-D ---
X_tsne = TSNE(
    n_components=2,
    perplexity=30,           # effective neighborhood size — Example B sweeps this
    init='pca',              # PCA initialization: more stable, reproducible layouts
    learning_rate='auto',    # modern recommended default
    random_state=RANDOM_SEED # t-SNE is stochastic; seed for reproducibility
).fit_transform(X_pca)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=digits.target, s=6,
                 cmap='tab10', alpha=0.7)
plt.colorbar(sc, label='True digit (never shown to t-SNE)')
plt.title('t-SNE map of digits — read WITHIN clusters, not BETWEEN them')
plt.tight_layout(); plt.show()

### Reading the output (Section 6, Example A)
- Roughly ten well-separated islands emerge, and coloring confirms each island is (mostly) one digit — t-SNE recovered the class structure with **zero label information**. This is why it's the standard first look at any embedding space (word vectors, image features, single-cell genomics).
- Look for the instructive imperfections: a few points sit inside the *wrong* island — genuinely ambiguous handwriting (a 9 drawn like a 4). t-SNE surfaces label noise and hard cases beautifully.
- Discipline reminders while you admire the picture: island-to-island gaps and island sizes are artifacts of the layout, not facts about the data. (For a faster, transform-capable alternative used heavily in industry, look up **UMAP** — the ideas transfer directly.)

In [ ]:
# ============================================================
# Example B: the perplexity sweep — what survives is what's real
# ============================================================
# Same data, three perplexities. Reading protocol: structure that appears in
# ALL THREE maps is a property of the data; structure that appears in only
# one is a property of that perplexity. (Subsampled to 600 points so the
# three runs take seconds, not minutes.)
from sklearn.manifold import TSNE

rng = np.random.default_rng(RANDOM_SEED)
sub = rng.choice(len(digits.data), 600, replace=False)
X_sub = StandardScaler().fit_transform(digits.data[sub])
X_sub_pca = PCA(n_components=30, random_state=RANDOM_SEED).fit_transform(X_sub)
y_sub = digits.target[sub]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, perp in zip(axes, [5, 30, 100]):
    emb = TSNE(n_components=2, perplexity=perp, init='pca',
               learning_rate='auto', random_state=RANDOM_SEED).fit_transform(X_sub_pca)
    ax.scatter(emb[:, 0], emb[:, 1], c=y_sub, s=8, cmap='tab10', alpha=0.8)
    ax.set_title(f'perplexity = {perp}')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Same 600 digits, three neighborhood sizes')
plt.tight_layout(); plt.show()

print("perplexity 5:   tiny neighborhoods -> clusters splinter into sub-islands;")
print("                at this setting t-SNE can manufacture 'clusters' even from")
print("                pure noise. Treat micro-structure with suspicion.")
print("perplexity 30:  the standard map — ten coherent islands.")
print("perplexity 100: ~1/6 of the whole dataset as 'neighborhood' -> local detail")
print("                is smoothed away, islands crowd toward one mass; push")
print("                perplexity toward n/3 and collapse follows.")
print("\nThe ten-digit island structure is visible in ALL THREE maps -> that part")
print("is data, not artifact. Everything that changed between panels is layout.")

### Reading the output (Section 6, Example B)
- The ten islands survive all three perplexities — **that invariance is your evidence** the digit structure is real. The sub-splintering at perplexity 5 and the crowding at 100 came and went with the knob — layout, not data.
- This is a general reading protocol for any stochastic embedding (t-SNE, UMAP): rerun across the key hyperparameter and across seeds; report only structure that persists. A single beautiful map is an anecdote.
- Note what the sweep cost: three runs on 600 points. On a million points, this discipline is expensive — one more reason industry reaches for UMAP (faster, and with a `.transform()` for new points).

### 🧠 Can you answer? (Section 6 — t-SNE)
1. **Write the shape of the two probability distributions t-SNE matches: Gaussian-kernel similarities $p_{ij}$ in the original space, Student-t similarities $q_{ij}$ in the embedding. What is the 'crowding problem,' and why do the t-distribution's heavy tails solve it?**
2. **The loss is $KL(P\|Q)$. Using KL's asymmetry, explain which mistake is punished harshly (true neighbors placed far apart) and which is barely punished (non-neighbors placed close) — and derive from this exactly why inter-cluster distances in the map mean nothing.**
3. **How is each point's Gaussian bandwidth $\sigma_i$ set from the perplexity value? What does perplexity ≈ effective number of neighbors mean, and what happens as perplexity approaches n (why does the map degenerate)?**
4. **At perplexity 5, t-SNE can render pure i.i.d. noise as convincing clusters. Explain the mechanism, and state the two-part protocol from Example B that protects you from being fooled.**
5. **What is early exaggeration (multiplying $p_{ij}$ in the first iterations), and why does it improve the *global* arrangement of clusters despite t-SNE's local objective?**
6. **Why is there no natural out-of-sample `transform()` — what would embedding one new point require re-solving? What do parametric t-SNE and UMAP do architecturally that restores a transform?**
7. **Two t-SNE runs with different seeds produce mirror-image maps with clusters in different corners. Are they 'different results'? Propose a quantitative way to compare embeddings that ignores rotation/reflection (e.g., neighborhood-preservation / trustworthiness metrics).**
8. **A colleague computes K-Means on t-SNE's 2-D output and reports beautiful silhouette scores. Give two distinct reasons this is methodologically rotten, then name the ONE legitimate use of clustering a t-SNE map.**
9. **Contrast t-SNE and UMAP in two sentences of mechanism (probability matching + KL vs fuzzy simplicial sets + cross-entropy), then list the practical deltas: speed, global structure retention, transform(), and which caveats survive unchanged in UMAP.**

## 7. Isolation Forest — Anomaly Detection
**Dataset Target:** Synthetic Transaction Amounts with Injected Fraud

### The concept, in plain words
Most anomaly detectors model "normal" and flag what's far from it. Isolation Forest flips the logic with one elegant observation: **anomalies are easier to isolate.** Build a tree by splitting on random features at random thresholds; a point in the dense heart of the data needs many splits before it sits alone in a leaf, while an outlier gets cut off in just a few. Average the isolation depth over many random trees → shallow average depth = anomaly.

Why it's popular in practice: no distance metric (so no scaling worries), handles mixed feature scales, roughly linear time, and works purely from feature geometry — **no fraud labels needed to train**.

### The one parameter that matters: `contamination`
Your prior estimate of the anomaly *rate* (here 3%). It doesn't change the anomaly *scores* — it sets the score threshold so that ~3% of points get flagged. Set it from domain knowledge (historical fraud rate). v2 described what happens if you inflate it; **Example B runs the sweep and puts numbers on it — then makes the fraud subtle and watches the perfect scores collapse.**

### Evaluation honesty
Training is unsupervised, but *evaluating* recall/precision requires ground truth. Here we injected the fraud ourselves so we can grade honestly; in production you'd grade against investigator-confirmed cases, discovered after the fact.

### Study Checklist
- [ ] Isolation Principle (anomalies need fewer random splits to isolate)
- [ ] Contamination Parameter — Prior on Anomaly Rate (Example B: sweep it)
- [ ] Anomaly Score Distribution Reading
- [ ] Precision/Recall on Injected Anomalies (when ground truth exists)
- [ ] Subtle Anomalies in the Overlap Zone — Where Perfect Scores Go to Die (Example B)
- [ ] Unsupervised in Training, Evaluable Only with Labels

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Example A: Isolation Forest Fraud Detection
# ============================================================
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(RANDOM_SEED)

# --- Step 1: synthesize transactions — 970 normal + 30 injected frauds ---
# Features: [transaction amount, hour of day]
normal = np.column_stack([
    rng.normal(50, 15, 970),     # typical purchases ~ $50
    rng.normal(14, 4, 970),      # mostly daytime hours
])
fraud = np.column_stack([
    rng.normal(400, 100, 30),                             # suspiciously large amounts
    rng.choice([2, 3, 4], 30) + rng.normal(0, 0.5, 30),   # 2-4 AM activity
])
X = np.vstack([normal, fraud])
y_true = np.array([0] * 970 + [1] * 30)   # ground truth — kept ONLY for grading

# --- Step 2: fit UNSUPERVISED (the model never sees y_true) ---
iso = IsolationForest(
    n_estimators=200,        # number of random isolation trees
    contamination=0.03,      # prior: 'we expect ~3% of rows to be anomalous'
    random_state=RANDOM_SEED,
).fit(X)

# .predict() returns +1 = normal, -1 = anomaly. Convert to 0/1 for grading.
pred = (iso.predict(X) == -1).astype(int)

# --- Step 3: grade against the injected ground truth ---
tp = ((pred == 1) & (y_true == 1)).sum()
print(f"Flagged as anomalies: {pred.sum()}  |  True frauds caught: {tp}/30")
print(f"Precision: {tp / max(pred.sum(), 1):.4f}   (flags that were real fraud)")
print(f"Recall:    {tp / 30:.4f}   (frauds that got flagged)")

# --- Step 4: the score landscape ---
# decision_function: positive = normal side, negative = anomaly side.
# Separated humps = an easy problem; overlapping humps = expect FP/FN trade-offs.
scores = iso.decision_function(X)
plt.figure(figsize=(8, 3.5))
plt.hist(scores[y_true == 0], bins=50, alpha=0.6, label='normal')
plt.hist(scores[y_true == 1], bins=20, alpha=0.8, label='injected fraud')
plt.axvline(0, color='red', ls='--', lw=1, label='decision threshold')
plt.legend(); plt.title('Isolation Forest Anomaly Scores')
plt.xlabel('decision_function score'); plt.tight_layout(); plt.show()

### Reading the output (Section 7, Example A)
- Perfect precision *and* recall — because we made the frauds cartoonishly obvious (8× normal amounts, 3 AM). The score histogram shows why: two humps with clear water between them. **Real fraud lives in the overlap zone**; Example B builds exactly that scenario.
- Connecting the two notebooks: anomaly detection sits at the boundary of the paradigms — unsupervised training, but any *quantitative* evaluation quietly borrows labels. Knowing which side of that line each claim stands on is the mark of someone who actually understands the field.

In [ ]:
# ============================================================
# Example B: the contamination dial, then fraud that fights back
# ============================================================
from sklearn.ensemble import IsolationForest

# --- Part 1: sweep contamination on the OBVIOUS fraud from Example A ---
# The scores never change — only the threshold moves. Watch the flagged count
# track contamination x n almost exactly, and precision/recall trade places.
# (Reuses X, y_true from Example A.)
print("== obvious fraud (true rate 3%) ==")
print("contamination | flagged | precision | recall")
for c in [0.01, 0.03, 0.10]:
    iso_c = IsolationForest(n_estimators=200, contamination=c,
                            random_state=RANDOM_SEED).fit(X)
    pred_c = (iso_c.predict(X) == -1).astype(int)
    tp = ((pred_c == 1) & (y_true == 1)).sum()
    print(f"{c:>13} | {pred_c.sum():>7} | {tp/max(pred_c.sum(),1):>9.3f} | {tp/30:.3f}")
print("0.01: too stingy -> 10 flags, catches only the 10 most extreme frauds")
print("      (recall 0.33). 0.10: too generous -> 100 flags, 70 of them innocent")
print("      customers (precision 0.30). The prior IS the precision/recall dial.\n")

# --- Part 2: subtle fraud — drawn from a distribution OVERLAPPING normal ---
# Same detector, same true rate (3%), but frauds now mimic normal behavior:
# amounts ~ N(110, 35) instead of N(400, 100), hours ~ N(11, 5) not 3 AM.
rng = np.random.default_rng(RANDOM_SEED)
fraud_subtle = np.column_stack([
    rng.normal(110, 35, 30),   # elevated but plausible amounts
    rng.normal(11, 5, 30),     # ordinary hours
])
X_sub = np.vstack([normal, fraud_subtle])
y_sub = np.array([0] * 970 + [1] * 30)

iso_s = IsolationForest(n_estimators=200, contamination=0.03,
                        random_state=RANDOM_SEED).fit(X_sub)
pred_s = (iso_s.predict(X_sub) == -1).astype(int)
tp_s = ((pred_s == 1) & (y_sub == 1)).sum()
print("== subtle fraud (same 3% rate, same detector) ==")
print(f"Precision: {tp_s/max(pred_s.sum(),1):.3f} | Recall: {tp_s/30:.3f}"
      "   <- down from 1.000 / 1.000")

# The score landscape explains everything: the humps now overlap.
scores_s = iso_s.decision_function(X_sub)
plt.figure(figsize=(8, 3.5))
plt.hist(scores_s[y_sub == 0], bins=50, alpha=0.6, label='normal')
plt.hist(scores_s[y_sub == 1], bins=20, alpha=0.8, label='subtle fraud')
plt.axvline(np.quantile(scores_s, 0.03), color='red', ls='--', lw=1,
            label='3% threshold')
plt.legend(); plt.title('Subtle fraud: the humps overlap — no threshold is clean')
plt.xlabel('decision_function score'); plt.tight_layout(); plt.show()

print("\nGeometry-only detection has hit its ceiling: frauds sitting inside the")
print("normal hump are invisible to ANY threshold. The production escalation")
print("path: engineer features that re-separate the humps (velocity: tx/hour,")
print("deviation from THIS customer's own baseline, merchant risk) and layer a")
print("supervised model trained on confirmed cases on top of this detector.")

### Reading the output (Section 7, Example B)
- The contamination sweep quantifies the prior's power: at 0.01 the detector catches only the 10 most extreme frauds (recall 0.33); at 0.10 it flags 100 points of which 70 are innocent (precision 0.30). The *scores* were identical in all three runs — you were only sliding a cutoff along them. Contamination is a business decision wearing a hyperparameter's clothes.
- One artifact worth noticing: with contamination set exactly to the true rate, flagged count = true fraud count, so **precision ≈ recall by construction** (any miss creates exactly one false alarm). If you see suspiciously equal precision and recall in an anomaly report, check whether the threshold was set to the true rate — that equality is arithmetic, not skill.
- The subtle-fraud panel is the real lesson: precision and recall fall to 0.60 not because the model got worse but because the *information isn't in these two features anymore*. Overlapping score humps are a feature-engineering problem, not a model-tuning problem — no threshold, ensemble size, or algorithm swap fixes them.

### 🧠 Can you answer? (Section 7 — Isolation Forest)
1. **Explain the isolation principle probabilistically: why is the expected number of random axis-parallel splits needed to isolate a point in a dense region larger than for an outlier? What role does the normalizing constant $c(n) \approx 2\ln(n-1) + \gamma$ (average BST path length) play in the anomaly score formula?**
2. **Why does Isolation Forest need no feature scaling, and more strongly — why is it invariant to any monotone transformation applied per feature? Which earlier algorithm family shares this property and why?**
3. **Prove to yourself and then state: changing `contamination` cannot change which points are *ranked* most anomalous. What does it change, exactly, and why does flagged count ≈ contamination × n?**
4. **In Example B, precision equaled recall in every run where contamination matched the true rate. Show the arithmetic that forces this, and explain what it implies about reading vendor anomaly-detection benchmarks.**
5. **The default `max_samples=256` means each tree sees only 256 random rows even from millions. Explain the *swamping* and *masking* effects that make small subsamples actively BETTER for isolation, not just faster.**
6. **This notebook fits and scores on the same data — outlier detection. Contrast with *novelty* detection (fit on clean data, score new data): which sklearn parameter/algorithms change, and in the fraud setting, which regime does a live transaction stream put you in?**
7. **Construct an anomaly Isolation Forest structurally struggles with: a point whose amount is normal AND whose hour is normal, but whose *combination* violates a strong correlation in the data. Why do axis-parallel random splits under-isolate it, and what does Extended Isolation Forest change?**
8. **A GMM's `score_samples` (Section 4) also yields anomaly scores. Compare the two detectors on: distributional assumptions, behavior on multi-modal normals, dimensionality tolerance, and what each considers 'anomalous' — then name a scenario where each clearly wins.**
9. **The evaluation used labels that were also present (unlabeled) in the training data. Is that leakage in the supervised sense? Argue precisely why not — then identify the one place an answer key DID quietly enter the pipeline (hint: who told contamination the true rate?), and how honest reporting handles it.**

## Mini Project Tracker

| Project | Topic | Dataset Benchmark | Model Baseline | Primary Metric | Status | Notes |
|---|---|---|---|---|---|---|
| **Customer Segmentation** | Clustering | Synthetic RFM | `KMeans(n_clusters=k)` | Inertia elbow + Silhouette + bootstrap stability ARI | Not Started | Profile clusters in original units; stability = the third lens |
| **Wine Grouping** | Hierarchical | load_wine | `AgglomerativeClustering(ward)` | Silhouette / ARI | Not Started | Compare linkages; cut by distance_threshold; see single-linkage chaining fail |
| **Non-Convex Shapes** | Density Clustering | make_moons | `DBSCAN(eps, min_samples)` | ARI + noise count | Not Started | k-distance elbow for eps; varying-density failure → HDBSCAN |
| **Overlapping Populations** | Probabilistic Clustering | make_blobs (varied std) | `GaussianMixture(full cov)` | BIC / soft membership | Not Started | BIC sweep for n_components; covariance types on sheared data; gmm.sample() |
| **Feature Compression** | Dim. Reduction | Breast Cancer + digits | `PCA(n_components=0.95)` | Explained Variance / reconstruction MSE | Not Started | Read PC1 loadings; visualize digit reconstructions at 5/15/40 comps |
| **Digits Map** | Visualization | load_digits | `PCA(30) -> TSNE(2)` | Visual cluster separation | Not Started | Perplexity sweep 5/30/100; trust only structure that survives all maps |
| **Fraud Flags** | Anomaly Detection | Synthetic Transactions | `IsolationForest(contamination)` | Precision/Recall on injected | Not Started | Contamination sweep; subtle-fraud overlap ceiling; feature engineering beats tuning |

## Experiment Log

| Date | Problem Context | Dataset Input | Model Configuration | Preprocessing Setup | Metric Tracked | Result | Next Progressive Steps |
|---|---|---|---|---|---|---|---|
| YYYY-MM-DD | Customer Segmentation | Synthetic RFM | KMeans (k=3, n_init=10) | StandardScaler | Silhouette + stability ARI | | Try k=4; compare with GMM soft labels |
| YYYY-MM-DD | Fraud Detection | Synthetic Transactions | IsolationForest (contam=0.03) | None (raw features) | Recall on injected | | Sweep contamination 0.01–0.10; add velocity feature for subtle fraud |